# BUSI — اجرای مستقل تنظیمات گزارش‌شدهٔ مقاله

**این فایل کد نویسندگان و بازتولید دقیق مقاله نیست.** از PDF به‌تنهایی نمی‌توان تمام تصمیم‌های لازم را بازیابی کرد. حالت پیش‌فرض عمداً اجازهٔ آموزش با فرض‌های تأییدنشده نمی‌دهد.

منبع: Islam et al., *Enhancing breast cancer segmentation and classification: An Ensemble Deep Convolutional Neural Network and U-net approach on ultrasound images*, 2024. DOI: [10.1016/j.mlwa.2024.100555](https://doi.org/10.1016/j.mlwa.2024.100555).

| تنظیم قطعی | مقدار مقاله | محل |
|---|---|---|
| ورودی | 224×224×3 | شکل ۶ / جدول ۲ |
| Optimizer | Adam | جدول ۲ |
| Learning rate | 0.001 ثابت در این پیاده‌سازی؛ scheduler در مقاله مشخص نشده | جدول ۲ |
| Batch | 32 | جدول ۲ |
| Epochs | 100 | جدول ۲ |
| Dropout | 0.4؛ محل دقیق مشخص نشده | جدول ۲ |
| Loss | SparseCategoricalCrossentropy | جدول ۲ |
| Metric | Accuracy | جدول ۲ |
| Normalization | Z-score؛ محور و جامعهٔ محاسبه مشخص نشده | بخش ۳٫۳ |
| Rotation / shifts / shear / zoom | 6 / 0.3 / 0.3 / 0.3 / 0.3 | جدول ۱ |
| Horizontal flip / fill | True / nearest | جدول ۱ |

## ابهام‌هایی که با حدس پنهان پر نشده‌اند

تعداد واحدهای Dense، نوع وزن‌های اولیه، وضعیت ثابت‌بودن شبکه‌ها، نسخهٔ دقیق MobileNet، محل Dropout، آمار Z-score، seed، تقسیم واقعی داده‌ها و انتخاب checkpoint روشن نیستند. تطبیق تعداد پارامترها به‌تنهایی معماری را اثبات نمی‌کند.

محاسبهٔ یک سرنخ: با Dense(128) روی نقشهٔ 7×7 و خروجی ۳کلاسه، تعداد پارامترها برابر `(C+1)*128 + (7*7*128+1)*3` است. برای MobileNetV1+Xception، C=3072 و نتیجه 412,163 است؛ برای MobileNetV2+Xception، C=3328 و نتیجه **444,931** است، دقیقاً عدد گزارش‌شدهٔ مقاله. **استفاده از V2 استنباط است، نه تصریح مقاله.**

## استفاده

در Colab یک Runtime دارای GPU انتخاب کن و Run all بزن. کتابخانه‌ها از محیط فعلی خوانده می‌شوند؛ نسخهٔ اصلی محیط مقاله منتشر نشده است.

- مقدار پیش‌فرض `ALLOW_DOCUMENTED_RECONSTRUCTION=False` آموزش را متوقف می‌کند، چون خواسته‌ای هیچ فرضی به مقاله نسبت داده نشود.
- فقط اگر بازسازی با فرض‌های آشکار را پذیرفتی، آن را `True` کن. آنگاه همین فایل به‌تنهایی آموزش را اجرا می‌کند؛ نیاز به نوت‌بوک‌های قبلی نیست.
- فرض‌های حالت بازسازی در کد و config ذخیره می‌شوند: V2، Dense128، ImageNet، شبکه‌های ثابت، Z-score کانالی فقط از train و Dropout بعد از Flatten. این انتخاب‌ها تضمین رسیدن به 87.82٪ نیستند.
- تقسیم پاک‌سازی‌شدهٔ قبلی حفظ می‌شود: 499/115/147. مقاله روی BUSI اصلی با 780 تصویر گزارش می‌دهد؛ این پروتکل یکسان نیست. test هرگز خودکار ارزیابی نمی‌شود.
- نه Fine-tuning، نه EarlyStopping، نه تغییر نرخ یادگیری، نه وزن‌دهی کلاس اضافه شده است. ۱۰۰ epoch اجرا می‌شود.
- بار دوم اجرا، دادهٔ cacheشده و آخرین checkpoint را می‌خواند؛ آموزش از صفر تکرار نمی‌شود. ادامه پس از قطع runtime به‌علت وضعیت تصادفی augmentation، بازپخش بیت‌به‌بیت اجرای بدون قطع نیست.

این فایل فقط آزمایش **طبقه‌بندی BUSI** را پوشش می‌دهد. مشخصات کافی برای U-Net دقیق نویسندگان یا آزمایش دیتاست دوم در دست نیست و معماری تازه‌ای به نام آن‌ها ساخته نشده است.

In [ ]:
# تنها تغییر اختیاری کاربر: اجازهٔ بازسازی با فرض‌های مستند
ALLOW_DOCUMENTED_RECONSTRUCTION = False

from pathlib import Path
ROOT = Path('/content/drive/MyDrive/BUSI_MERGED')
SPLIT_PATH = Path('/content/drive/MyDrive/BUSI_AUDIT_REPORTS/clean_rebuild_20261004T124959_768638Z/split_manifest.csv')
BASE_DIR = Path('/content/drive/MyDrive/BUSI_PAPER_SETTINGS')

print('Paper trainable parameters:', 444_931)
for name, channels in [('MobileNetV1 + Xception', 3072), ('MobileNetV2 + Xception', 3328)]:
    count = (channels + 1) * 128 + (7 * 7 * 128 + 1) * 3
    print(name, 'with inferred Dense128:', f'{count:,}')

if not ALLOW_DOCUMENTED_RECONSTRUCTION:
    raise RuntimeError(
        'STRICT MODE: تنظیمات ناقص مقاله با فرض پر نشده‌اند. '
        'برای بازتولید دقیق، کد/مدل و مشخصات پیش‌پردازش نویسندگان لازم است. '
        'اگر بازسازی با فرض‌های نوشته‌شده را می‌پذیری، '
        'ALLOW_DOCUMENTED_RECONSTRUCTION را True کن و Run all بزن. '
        'این تغییر، اجرای بازسازی را فعال می‌کند؛ آن را به بازتولید دقیق تبدیل نمی‌کند.'
    )


## اتصال و ثبت مشخصات
فقط داده‌های train و validation خوانده می‌شوند. فایل تقسیم، پارامترها و نسخهٔ محیط در پوشهٔ مستقل ذخیره می‌شوند.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, hashlib, math, platform, tempfile
import numpy as np
import pandas as pd
import tensorflow as tf
import PIL
from PIL import Image
from IPython.display import display
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42                         # engineering choice, not reported by paper
tf.keras.utils.set_random_seed(SEED)
tf.keras.backend.clear_session()
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('در Colab از Runtime > Change runtime type یک GPU انتخاب کن.')

assert ROOT.is_dir(), f'Dataset not found: {ROOT}'
assert SPLIT_PATH.is_file(), f'Fixed split not found: {SPLIT_PATH}'
manifest_bytes = SPLIT_PATH.read_bytes()
manifest_hash = hashlib.sha256(manifest_bytes).hexdigest()
df = pd.read_csv(SPLIT_PATH)
required = {'image', 'class_name', 'split', 'group_id'}
assert required <= set(df), f'Missing columns: {required - set(df)}'
assert len(df) == 761 and df['image'].is_unique
assert set(df['class_name']) == {'normal', 'benign', 'malignant'}
assert set(df['split']) == {'train', 'val', 'test'}
assert df['group_id'].notna().all()
assert df.groupby('group_id')['split'].nunique().max() == 1, 'Group leakage detected'
expected_counts = {'train': 499, 'val': 115, 'test': 147}
assert df['split'].value_counts().to_dict() == expected_counts
labels = {'normal': 0, 'benign': 1, 'malignant': 2}
df['label'] = df['class_name'].map(labels).astype('int32')
assert not any(Path(p).is_absolute() or '..' in Path(p).parts for p in df['image'])
display(pd.crosstab(df['split'], df['class_name'], margins=True))

config = {
    'status': 'documented_reconstruction_not_exact_replication',
    'paper_doi': '10.1016/j.mlwa.2024.100555',
    'reported_settings': {
        'image_size': [224, 224, 3], 'optimizer': 'Adam',
        'learning_rate': 0.001, 'batch_size': 32, 'epochs': 100,
        'dropout': 0.4, 'loss': 'SparseCategoricalCrossentropy',
        'metric': 'accuracy', 'normalization': 'zscore',
        'augmentation': {
            'rotation_range': 6, 'width_shift_range': 0.3,
            'height_shift_range': 0.3, 'shear_range': 0.3,
            'zoom_range': 0.3, 'horizontal_flip': True,
            'fill_mode': 'nearest'
        }
    },
    'unreported_choices_explicitly_assumed': {
        'mobile_variant': 'MobileNetV2_alpha1',
        'rationale': 'matches reported 444931 trainable parameters with Dense128; not proof',
        'weights': 'imagenet', 'backbones_trainable': False,
        'dense_units': 128, 'head': 'concat_maps_dense_relu_flatten_dropout_softmax',
        'dropout_position': 'after_flatten',
        'zscore': 'per_channel_training_images_only_before_augmentation',
        'resize': 'PIL_RGB_bilinear',
        'augmentation_api': 'tf.keras.preprocessing.image.ImageDataGenerator',
        'augmentation_units': 'legacy_Keras_rotation_and_shear_degrees_shifts_fractions',
        'augmentation_interpolation_order': 1,
        'augmentation_probability_horizontal_flip': 0.5,
        'adam_beta1': 0.9, 'adam_beta2': 0.999, 'adam_epsilon': 1e-7,
        'seed': SEED, 'label_order': labels,
        'checkpoint_selection': 'save_final_plus_best_val_loss_and_best_val_accuracy',
        'training_steps': 'ceil_train_count_divided_by_32_include_last_partial_batch'
    },
    'protocol_difference': 'existing_cleaned_grouped_split_761_not_original_780',
    'split_manifest': str(SPLIT_PATH), 'split_sha256': manifest_hash,
    'versions': {'tensorflow': tf.__version__, 'keras': tf.keras.__version__,
                 'python': platform.python_version(), 'numpy': np.__version__,
                 'pillow': PIL.__version__}
}
config_text = json.dumps(config, sort_keys=True, ensure_ascii=False)
run_key = hashlib.sha256(config_text.encode()).hexdigest()[:16]
run_dir = BASE_DIR / ('reconstruction_' + run_key)
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / 'config.json').write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf8')
if not (run_dir / 'split_manifest.csv').exists():
    (run_dir / 'split_manifest.csv').write_bytes(manifest_bytes)
print('TensorFlow:', tf.__version__, '| GPU:', gpus)
print('Run folder:', run_dir)
print('این اجرا بازسازی مستند است، نه کد دقیق نویسندگان.')


## بارگذاری یک‌باره و Z-score
cache فقط سرعت خواندن را بهتر می‌کند؛ نمونه‌ها یا تقسیم داده را تغییر نمی‌دهد. آمار کانالی فقط از train محاسبه می‌شود؛ جامعهٔ محاسبه در مقاله مشخص نشده و این انتخاب در config ثبت شده است.

In [ ]:
cache_dir = BASE_DIR / 'cache'
cache_dir.mkdir(parents=True, exist_ok=True)
cache_recipe = f'{manifest_hash}|PIL-{PIL.__version__}|RGB|224|BILINEAR|v1'
cache_key = hashlib.sha256(cache_recipe.encode()).hexdigest()[:16]
cache_path = cache_dir / f'classification_{cache_key}.npz'
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)

def load_images(frame, title):
    images = np.empty((len(frame), 224, 224, 3), dtype=np.uint8)
    for i, rel in enumerate(frame['image']):
        with Image.open(ROOT / rel) as im:
            images[i] = np.asarray(im.convert('RGB').resize((224, 224), Image.Resampling.BILINEAR))
        if i % 100 == 0:
            print(f'{title}: {i+1}/{len(frame)}')
    return images

if cache_path.is_file():
    with np.load(cache_path, allow_pickle=False) as data:
        X_train_u8, X_val_u8 = data['train'], data['val']
    print('Loaded cached train/validation images.')
else:
    X_train_u8 = load_images(train_df, 'Train')
    X_val_u8 = load_images(val_df, 'Validation')
    temporary_cache = cache_path.with_suffix('.tmp.npz')
    np.savez_compressed(temporary_cache, train=X_train_u8, val=X_val_u8)
    os.replace(temporary_cache, cache_path)

assert X_train_u8.shape == (499,224,224,3)
assert X_val_u8.shape == (115,224,224,3)
y_train = train_df['label'].to_numpy(dtype=np.int32)
y_val = val_df['label'].to_numpy(dtype=np.int32)
mean = X_train_u8.mean(axis=(0,1,2), dtype=np.float64).astype('float32')
std = X_train_u8.std(axis=(0,1,2), dtype=np.float64).astype('float32')
assert np.all(std > 0)
X_train = (X_train_u8.astype('float32') - mean) / std
X_val = (X_val_u8.astype('float32') - mean) / std
del X_train_u8, X_val_u8
(run_dir / 'normalization.json').write_text(json.dumps({'mean': mean.tolist(), 'std': std.tolist()}, indent=2))
print('Training mean:', mean, '\nTraining std:', std)
print('Train:', X_train.shape, '| Validation:', X_val.shape)


## augmentation با تمام اعداد جدول ۱
در این بازسازی، پارامترها به API قدیمی Keras داده می‌شوند؛ ۶ یعنی درجهٔ چرخش و ۰٫۳ برای shear نیز در این API درجه است، نه ۳۰٪. مقاله API و واحد shear را مشخص نکرده است. جابه‌جایی ۳۰٪ و زوم می‌توانند بخشی از ضایعه را از تصویر خارج کنند؛ این مقادیر را برای بهترشدن نتیجه تغییر نداده‌ایم. validation تغییر تصادفی ندارد.

In [ ]:
aug_settings = config['reported_settings']['augmentation']
train_generator = ImageDataGenerator(**aug_settings, interpolation_order=1)
train_flow = train_generator.flow(
    X_train, y_train, batch_size=32, shuffle=True, seed=SEED
)

# نمایش با نمونهٔ جدا؛ وضعیت تصادفی مولد آموزش مصرف نمی‌شود
import matplotlib.pyplot as plt
preview_generator = ImageDataGenerator(**aug_settings, interpolation_order=1)
fig, axes = plt.subplots(3, 2, figsize=(8, 10))
for row, cls in enumerate(['normal', 'benign', 'malignant']):
    index = int(np.flatnonzero(train_df['class_name'].to_numpy() == cls)[0])
    image = X_train[index]
    transformed = preview_generator.random_transform(image.copy(), seed=SEED + row)
    for col, values in enumerate([image, transformed]):
        shown = np.clip((values * std + mean) / 255., 0, 1)
        axes[row, col].imshow(shown)
        axes[row, col].set_title(cls + (' / original' if col == 0 else ' / augmented'))
        axes[row, col].axis('off')
plt.tight_layout()
fig.savefig(run_dir / 'augmentation_preview.png', dpi=120)
plt.show()


## مدل بازسازی‌شده و ادامهٔ خودکار
V2+Xception+Dense128 با شبکه‌های ثابت، دقیقاً ۴۴۴٬۹۳۱ پارامتر قابل‌آموزش می‌دهد. این تطبیق فقط یک شاهد سازگاری است. به‌جای تغییر Z-score به preprocessing استاندارد ImageNet، همان Z-score گزارش‌شده اعمال می‌شود؛ نتیجهٔ بهتر تضمین نمی‌شود.

`last.keras` وضعیت optimizer و شمارندهٔ آن را نگه می‌دارد. log باید با شمارهٔ epoch ذخیره‌شده سازگار باشد؛ در صورت قطع هنگام ذخیره، فایل‌ها خودکار پاک یا بازنویسی نمی‌شوند.

In [ ]:
def save_atomic(model, path):
    path = Path(path)
    tmp = path.with_name(path.stem + '.tmp.keras')
    model.save(str(tmp))
    os.replace(tmp, path)

log_path = run_dir / 'training_log.csv'
last_path = run_dir / 'last.keras'
if last_path.exists():
    model = tf.keras.models.load_model(str(last_path))
    steps = math.ceil(len(X_train) / 32)
    updates = int(model.optimizer.iterations.numpy())
    assert updates % steps == 0, 'Checkpoint is not at an epoch boundary.'
    initial_epoch = updates // steps
    if log_path.exists():
        log = pd.read_csv(log_path)
        assert log['epoch'].tolist() == list(range(initial_epoch)), 'CSV/checkpoint mismatch; preserve files and investigate.'
    else:
        assert initial_epoch == 0, 'Training log missing; preserve checkpoint and investigate.'
    print('Resuming at epoch:', initial_epoch)
else:
    assert not log_path.exists(), 'Log exists without checkpoint; investigate before restarting.'
    raw = tf.keras.Input((224,224,3), name='zscore_image')
    mobile = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(224,224,3))
    xception = tf.keras.applications.Xception(include_top=False, weights='imagenet', input_shape=(224,224,3))
    mobile.trainable = False
    xception.trainable = False
    features = tf.keras.layers.Concatenate(name='feature_concat')([
        mobile(raw, training=False), xception(raw, training=False)
    ])
    x = tf.keras.layers.Dense(128, activation='relu', name='dense_features')(features)
    x = tf.keras.layers.Flatten(name='flatten')(x)
    x = tf.keras.layers.Dropout(0.4, name='dropout')(x)
    output = tf.keras.layers.Dense(3, activation='softmax', name='class_probabilities')(x)
    model = tf.keras.Model(raw, output, name='parameter_matched_reconstruction')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-7),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy']
    )
    initial_epoch = 0

trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
assert trainable == 444_931, f'Unexpected trainable parameters: {trainable}'
print('Total parameters:', model.count_params(), '| Trainable:', trainable)

class PersistentCheckpoints(tf.keras.callbacks.Callback):
    def __init__(self, loss_best, accuracy_best):
        super().__init__()
        self.loss_best = loss_best
        self.accuracy_best = accuracy_best
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if not np.isfinite(float(logs.get('loss', np.nan))) or not np.isfinite(float(logs.get('val_loss', np.nan))):
            raise FloatingPointError('Non-finite loss: stop and inspect; do not report this run as successful.')
        if logs['val_loss'] < self.loss_best:
            save_atomic(self.model, run_dir / 'best_val_loss.keras')
            self.loss_best = float(logs['val_loss'])
        if logs['val_accuracy'] > self.accuracy_best:
            save_atomic(self.model, run_dir / 'best_val_accuracy.keras')
            self.accuracy_best = float(logs['val_accuracy'])
        save_atomic(self.model, last_path)

old_log = pd.read_csv(log_path) if log_path.exists() else pd.DataFrame()
best_loss = float(old_log['val_loss'].min()) if len(old_log) else float('inf')
best_accuracy = float(old_log['val_accuracy'].max()) if len(old_log) else -float('inf')

if initial_epoch < 100:
    model.fit(
        train_flow, validation_data=(X_val, y_val),
        epochs=100, initial_epoch=initial_epoch,
        callbacks=[
            PersistentCheckpoints(best_loss, best_accuracy),
            tf.keras.callbacks.CSVLogger(str(log_path), append=initial_epoch > 0)
        ]
    )
else:
    print('100 epochs already complete. Training skipped.')


## گزارش validation؛ بدون استفاده از test
چون سیاست انتخاب مدل در مقاله روشن نیست، نتایج آخرین epoch و دو سیاست انتخاب ذخیره می‌شوند. هیچ‌کدام به‌عنوان «سیاست نویسندگان» معرفی نمی‌شود. Accuracy مقاله روی test است؛ این جدول روی validation است و مقایسهٔ هم‌شرایط محسوب نمی‌شود.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
class_names = ['normal', 'benign', 'malignant']
rows = []
for checkpoint in ['last', 'best_val_loss', 'best_val_accuracy']:
    candidate = tf.keras.models.load_model(str(run_dir / f'{checkpoint}.keras'), compile=False)
    probabilities = candidate.predict(X_val, batch_size=32, verbose=0)
    prediction = np.argmax(probabilities, axis=1)
    report = classification_report(y_val, prediction, labels=[0,1,2], target_names=class_names, output_dict=True, zero_division=0)
    matrix = confusion_matrix(y_val, prediction, labels=[0,1,2])
    pd.DataFrame(report).T.to_csv(run_dir / f'{checkpoint}_validation_report.csv')
    pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv(run_dir / f'{checkpoint}_validation_confusion.csv')
    frame = val_df[['image', 'class_name', 'group_id']].copy()
    frame['predicted_label'] = prediction
    for i, name in enumerate(class_names):
        frame[f'p_{name}'] = probabilities[:, i]
    frame.to_csv(run_dir / f'{checkpoint}_validation_predictions.csv', index=False)
    rows.append({'checkpoint': checkpoint, 'validation_accuracy': accuracy_score(y_val, prediction),
                 'macro_f1': f1_score(y_val, prediction, labels=[0,1,2], average='macro', zero_division=0),
                 'normal_recall': report['normal']['recall'], 'malignant_recall': report['malignant']['recall']})
    del candidate
summary = pd.DataFrame(rows)
summary.to_csv(run_dir / 'validation_summary.csv', index=False)
display(summary)
print('Saved:', run_dir)
print('Test images have not been loaded or evaluated.')
